In [ ]:
dataset_path = ''
target_column = 'revenue'
output_dir = ''
dataset_version_id = ''
random_seed = 42
currency = 'USD'
project_root = ''


In [ ]:
if dataset_path:
    import json
    from pathlib import Path
    import joblib
    import lightgbm as lgb
    import numpy as np
    import pandas as pd
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    import sys
    if project_root:
        sys.path.insert(0, project_root)
    from app.ml.genres import MultiHotGenreEncoder, parse_genres

    if target_column != "revenue":
        raise ValueError("This adapter currently trains the revenue target only")
    data = pd.read_csv(dataset_path)
    required = {"budget", "genre", "revenue"}
    missing = required - set(data.columns)
    if missing:
        raise ValueError(f"Missing required training columns: {sorted(missing)}")
    if len(data) < 10:
        raise ValueError("At least 10 standardized rows are required for LightGBM training")
    feature_columns = [name for name in ["budget", "genre"]
                       if name in data.columns and data[name].notna().any()]
    numeric_columns = [name for name in feature_columns if name != "genre"]
    X = data[feature_columns].copy()
    y = pd.to_numeric(data[target_column], errors="coerce")
    valid = y.notna() & np.isfinite(y)
    X, y = X.loc[valid], y.loc[valid]
    if len(X) < 10:
        raise ValueError("At least 10 rows with numeric revenue targets are required")
    X_train, X_holdout, y_train, y_holdout = train_test_split(
        X, y, test_size=0.2, random_state=int(random_seed)
    )
    genre_vocabulary = []
    seen_genres = set()
    for value in data['genre'].dropna():
        for genre_name in parse_genres(value):
            if genre_name.casefold() not in seen_genres:
                genre_vocabulary.append(genre_name)
                seen_genres.add(genre_name.casefold())
    preprocessor = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), numeric_columns),
        ("genre", MultiHotGenreEncoder(categories=genre_vocabulary), ["genre"]),
    ])
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", lgb.LGBMRegressor(
            objective="regression", n_estimators=500, learning_rate=0.03,
            num_leaves=15, max_depth=6, min_child_samples=max(2, min(20, len(X_train) // 10)),
            subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            reg_alpha=0.5, reg_lambda=2.0, random_state=int(random_seed),
            n_jobs=1, verbosity=-1,
        )),
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_holdout)
    metrics = {
        "mae": float(mean_absolute_error(y_holdout, predictions)),
        "rmse": float(np.sqrt(mean_squared_error(y_holdout, predictions))),
        "holdout_rows": int(len(y_holdout)),
    }
    manifest = {
        "model_type": "lgbm_revenue", "artifact_version": "1",
        "training_dataset_version": dataset_version_id, "target": target_column,
        "feature_columns": feature_columns, "output_type": "point_prediction",
        "model_file": "model.joblib", "currency": currency or "USD",
        "genre_vocabulary": genre_vocabulary, "supports_multiple_genres": True,
        "genre_encoding": "multi_hot_v1",
        "estimator": "lightgbm.LGBMRegressor",
    }
    folder = Path(output_dir)
    folder.mkdir(parents=True, exist_ok=True)
    joblib.dump(pipeline, folder / "model.joblib")
    (folder / "metrics.json").write_text(json.dumps(metrics))
    (folder / "manifest.json").write_text(json.dumps(manifest))
    print("LightGBM training complete:", metrics)
else:
    # =====================================================================================
    #  MOVIE PREDICTOR (LightGBM) - Estimasi Penjualan & Rating Film
    #  Flow: Load -> EDA -> Cleaning -> EDA visual -> Split 70/15/15 -> Feature Engineering
    #        -> Training (early stopping) -> Evaluasi -> Feature Importance -> Output Prediksi
    # =====================================================================================
    import os, re, glob, warnings
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import lightgbm as lgb
    import joblib
    from sklearn.model_selection import train_test_split, KFold
    from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
    try:
        from IPython.display import display
    except Exception:
        display = print

    warnings.filterwarnings("ignore")
    sns.set_theme(style="whitegrid", context="notebook")
    pd.set_option("display.max_columns", 60, "display.width", 220,
                  "display.float_format", lambda v: f"{v:,.2f}")
    SEED = 42
    np.random.seed(SEED)

    DATA_DIR = "/kaggle/input/datasets/mjshubham21/movie-dataset-for-analytics-and-visualization"
    OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./output"
    os.makedirs(OUT_DIR, exist_ok=True)

    # Target yang diprediksi: "money" -> di-log1p saat training, "rating" -> skala asli
    TARGETS = {
        "Global_BoxOfficeUSD": "money", "US_BoxOfficeUSD": "money",
        "Opening_Day_SalesUSD": "money", "One_Week_SalesUSD": "money",
        "IMDbRating": "rating", "RottenTomatoesScore": "rating",
    }
    BOUNDS = {"IMDbRating": (0, 10), "RottenTomatoesScore": (0, 100)}
    NUM_COLS = ["ReleaseYear", "BudgetUSD", "US_BoxOfficeUSD", "Global_BoxOfficeUSD", "Opening_Day_SalesUSD",
                "One_Week_SalesUSD", "IMDbRating", "RottenTomatoesScore", "NumVotesIMDb", "NumVotesRT"]
    TXT_COLS = ["Title", "Genre", "Country", "Director", "LeadActor"]


    # ------------------------------------------------------------------ helper umum
    def section(t):
        print("\n" + "=" * 100 + f"\n{t}\n" + "=" * 100)

    def read_csv_safe(f, **kw):
        try:
            return pd.read_csv(f, **kw)
        except UnicodeDecodeError:
            return pd.read_csv(f, encoding="latin-1", **kw)

    def parse_genres(g):
        if pd.isna(g):
            return ["Unknown"]
        parts = [p.strip().title() for p in re.split(r"[,|/;&]+", str(g)) if p.strip()]
        return parts or ["Unknown"]

    def to_num(s):
        if not pd.api.types.is_numeric_dtype(s):
            s = s.astype(str).str.replace(r"[\$,%\s]", "", regex=True)
            s = s.mask(s.str.lower().isin(["", "nan", "none", "n/a", "na", "null"]))
        return pd.to_numeric(s, errors="coerce")

    def san(g):  # nama fitur aman untuk LightGBM
        return re.sub(r"[^0-9a-zA-Z_]", "_", g)

    def to_orig(pred, tgt, kind):
        pred = np.asarray(pred, dtype=float)
        return np.clip(np.expm1(pred), 0, None) if kind == "money" else np.clip(pred, *BOUNDS[tgt])

    def metrics(y_true, y_pred, kind):
        y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
        a, b = (np.log1p(y_true), np.log1p(y_pred)) if kind == "money" else (y_true, y_pred)
        return {"R2": r2_score(a, b),
                "RMSE": float(np.sqrt(mean_squared_error(a, b))),
                "MAE": mean_absolute_error(y_true, y_pred),
                "MedAPE_%": float(np.median(np.abs(y_true - y_pred) / np.maximum(np.abs(y_true), 1e-9)) * 100)}


    # =====================================================================================
    # 1. LOAD DATA
    # =====================================================================================
    section("1. LOAD DATA")
    def find_csv():
        cands = glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True) + \
                glob.glob("/kaggle/input/**/*.csv", recursive=True)
        for f in dict.fromkeys(cands):
            try:
                cols = set(read_csv_safe(f, nrows=0).columns.str.strip())
                if {"Title", "BudgetUSD", "Global_BoxOfficeUSD"} <= cols:
                    return f
            except Exception:
                continue
        raise FileNotFoundError("CSV film tidak ditemukan. Cek DATA_DIR / pastikan dataset sudah di-Add di Kaggle.")

    CSV_PATH = find_csv()
    raw = read_csv_safe(CSV_PATH)
    raw.columns = raw.columns.str.strip()
    print("File :", CSV_PATH, "| shape:", raw.shape)
    display(raw.head())


    # =====================================================================================
    # 2. EDA AWAL (data mentah)
    # =====================================================================================
    section("2. EDA AWAL (data mentah)")
    eda = pd.DataFrame({"dtype": raw.dtypes.astype(str), "missing": raw.isna().sum(),
                        "missing_%": raw.isna().mean() * 100, "n_unique": raw.nunique()})
    display(eda)
    print("Baris duplikat :", raw.duplicated().sum(),
          "| MovieID duplikat:", raw["MovieID"].duplicated().sum() if "MovieID" in raw else "-")
    display(raw.describe(include="all").T)


    # =====================================================================================
    # 3. DATA CLEANING
    # =====================================================================================
    section("3. DATA CLEANING")
    df = raw.copy()
    for c in NUM_COLS + TXT_COLS + ["ReleaseDate"]:
        if c not in df:
            df[c] = np.nan                                   # jaga-jaga kolom hilang
    for c in NUM_COLS:
        df[c] = to_num(df[c])                                # "$1,000" / "85%" -> angka
    for c in TXT_COLS:                                       # trim spasi & samakan nilai kosong
        s = df[c].astype(str).str.strip()
        df[c] = s.mask(s.str.lower().isin(["", "nan", "none", "n/a", "na", "null", "unknown"]))
    try:
        df["ReleaseDate"] = pd.to_datetime(df["ReleaseDate"], errors="coerce", format="mixed")
    except Exception:
        df["ReleaseDate"] = pd.to_datetime(df["ReleaseDate"], errors="coerce")

    n0 = len(df)
    df = df.drop_duplicates()
    if "MovieID" in df and df["MovieID"].notna().any():
        df = df.drop_duplicates(subset="MovieID")
    df = df.dropna(subset=list(TARGETS), how="all")          # buang baris tanpa target sama sekali

    # nilai tidak valid -> NaN (LightGBM menangani NaN secara native)
    df.loc[df["BudgetUSD"] <= 0, "BudgetUSD"] = np.nan
    for t, k in TARGETS.items():
        if k == "money":
            df.loc[df[t] <= 0, t] = np.nan
    df.loc[~df["IMDbRating"].between(1, 10), "IMDbRating"] = np.nan
    df.loc[~df["RottenTomatoesScore"].between(0, 100), "RottenTomatoesScore"] = np.nan
    df.loc[~df["ReleaseYear"].between(1880, 2100), "ReleaseYear"] = np.nan
    df["ReleaseYear"] = df["ReleaseYear"].fillna(df["ReleaseDate"].dt.year)
    df = df.reset_index(drop=True)

    # aturan konsistensi (A >= B) hanya dipakai bila memang berlaku di >= 95% data
    ORDER_RULES = []
    for big, small in [("Global_BoxOfficeUSD", "US_BoxOfficeUSD"), ("One_Week_SalesUSD", "Opening_Day_SalesUSD")]:
        both = df[[big, small]].dropna()
        if len(both) and (both[big] >= both[small]).mean() >= 0.95:
            ORDER_RULES.append((big, small))

    money_cols = [t for t, k in TARGETS.items() if k == "money"]
    print(f"Baris: {n0} -> {len(df)} (duplikat/tanpa-target dibuang)")
    print("Aturan konsistensi output (A >= B):", ORDER_RULES or "tidak ada")
    print("\nSkewness kolom uang (mentah vs log1p) -> alasan memakai transformasi log:")
    display(pd.DataFrame({"skew_mentah": df[money_cols + ["BudgetUSD"]].skew(),
                          "skew_log1p": np.log1p(df[money_cols + ["BudgetUSD"]]).skew()}))
    print("Missing value tersisa (dibiarkan NaN; kategori kosong -> 'Unknown'):")
    display(df.isna().sum()[lambda s: s > 0].to_frame("missing"))


    # =====================================================================================
    # 4. EDA VISUAL (data bersih)
    # =====================================================================================
    section("4. EDA VISUAL")
    df["_g1"] = df["Genre"].apply(lambda g: parse_genres(g)[0])

    fig, ax = plt.subplots(2, 3, figsize=(18, 9))
    sns.histplot(np.log10(df["BudgetUSD"].dropna()), bins=40, kde=True, ax=ax[0, 0]).set_title("log10(Budget USD)")
    sns.histplot(np.log10(df["Global_BoxOfficeUSD"].dropna()), bins=40, kde=True, ax=ax[0, 1]).set_title("log10(Global Box Office USD)")
    sns.histplot(df["IMDbRating"].dropna(), bins=30, kde=True, ax=ax[0, 2]).set_title("IMDb Rating")
    sns.histplot(df["RottenTomatoesScore"].dropna(), bins=30, kde=True, ax=ax[1, 0]).set_title("Rotten Tomatoes Score")
    sns.scatterplot(x=np.log10(df["BudgetUSD"]), y=np.log10(df["Global_BoxOfficeUSD"]), alpha=.35, ax=ax[1, 1])
    ax[1, 1].set_title("Budget vs Global Box Office (log10)")
    sns.scatterplot(x=df["IMDbRating"], y=df["RottenTomatoesScore"], alpha=.35, ax=ax[1, 2])
    ax[1, 2].set_title("IMDb vs Rotten Tomatoes")
    plt.tight_layout(); plt.show()

    top_g = df["_g1"].value_counts().head(12).index
    fig, ax = plt.subplots(2, 2, figsize=(18, 11))
    cdf = df[[c for c in NUM_COLS if c in df]].copy()
    for c in cdf.columns:
        if c != "ReleaseYear" and cdf[c].max() > 1000:
            cdf[c] = np.log1p(cdf[c])
    sns.heatmap(cdf.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, annot_kws={"size": 7}, ax=ax[0, 0])
    ax[0, 0].set_title("Korelasi (kolom uang & votes dalam log1p)")
    (df[df["_g1"].isin(top_g)].groupby("_g1")["Global_BoxOfficeUSD"].median().sort_values()
     .plot.barh(ax=ax[0, 1], title="Median Global Box Office per Genre Utama"))
    yr = df.groupby("ReleaseYear")[["BudgetUSD", "Global_BoxOfficeUSD"]].median()
    np.log10(yr).plot(ax=ax[1, 0], title="Tren median per tahun (log10 USD)")
    (df[df["_g1"].isin(top_g)].groupby("_g1")["IMDbRating"].mean().sort_values()
     .plot.barh(ax=ax[1, 1], title="Rata-rata IMDb Rating per Genre Utama"))
    plt.tight_layout(); plt.show()
    df = df.drop(columns="_g1")


    # =====================================================================================
    # 5. SPLIT DATA: Train 70% / Validation 15% / Test 15% (stratified)
    #    Dilakukan SEBELUM feature engineering agar statistik fitur hanya dipelajari dari train (no leakage)
    # =====================================================================================
    section("5. SPLIT DATA (70 / 15 / 15)")
    strat_src = np.log1p(df["Global_BoxOfficeUSD"])
    strat_src = strat_src.fillna(strat_src.median())
    all_idx = df.index.to_numpy()
    try:   # stratifikasi pada desil box office agar sebaran target di 3 subset seimbang
        strat = pd.qcut(strat_src, q=10, labels=False, duplicates="drop")
        idx_tr, idx_tmp = train_test_split(all_idx, test_size=0.30, random_state=SEED, stratify=strat)
        idx_va, idx_te = train_test_split(idx_tmp, test_size=0.50, random_state=SEED, stratify=strat.loc[idx_tmp])
        print("Metode split: stratified (desil Global Box Office)")
    except ValueError:
        idx_tr, idx_tmp = train_test_split(all_idx, test_size=0.30, random_state=SEED)
        idx_va, idx_te = train_test_split(idx_tmp, test_size=0.50, random_state=SEED)
        print("Metode split: random (data terlalu kecil untuk stratifikasi)")
    idx_tr, idx_va, idx_te = pd.Index(idx_tr), pd.Index(idx_va), pd.Index(idx_te)
    bal = pd.DataFrame({s: {"n_baris": len(i),
                            "mean_log_GlobalBO": strat_src.loc[i].mean(),
                            "mean_log_Budget": np.log1p(df.loc[i, "BudgetUSD"]).mean(),
                            "mean_IMDb": df.loc[i, "IMDbRating"].mean(),
                            "mean_RT": df.loc[i, "RottenTomatoesScore"].mean()}
                        for s, i in [("train", idx_tr), ("valid", idx_va), ("test", idx_te)]}).T
    print("Cek keseimbangan distribusi antar subset:")
    display(bal)


    # =====================================================================================
    # 6. FEATURE ENGINEERING (statistik dipelajari dari TRAIN saja)
    #    Hanya fitur yang sudah diketahui SEBELUM film rilis (NumVotes & rating/penjualan lain TIDAK dipakai = anti leakage)
    # =====================================================================================
    section("6. FEATURE ENGINEERING")
    tr_df = df.loc[idx_tr]
    min_n = max(5, int(0.01 * len(tr_df)))

    genre_cnt = pd.Series([g for gl in tr_df["Genre"].apply(parse_genres) for g in set(gl)]).value_counts()
    TOP_GENRES = [g for g in genre_cnt[genre_cnt >= min_n].index if g != "Unknown"][:20]
    PRIMARY_CATS = list(dict.fromkeys(TOP_GENRES + ["Unknown", "Other"]))
    ctry_cnt = tr_df["Country"].fillna("Unknown").value_counts()
    COUNTRY_CATS = list(dict.fromkeys(ctry_cnt[ctry_cnt >= min_n].index[:15].tolist() + ["Unknown", "Other"]))
    YEAR_BUDGET_MED = np.log1p(tr_df["BudgetUSD"]).groupby(tr_df["ReleaseYear"]).median()
    GLOBAL_BUDGET_MED = float(np.log1p(tr_df["BudgetUSD"]).median())
    DIRECTOR_FREQ = tr_df["Director"].fillna("Unknown").value_counts()
    ACTOR_FREQ = tr_df["LeadActor"].fillna("Unknown").value_counts()

    def build_base_features(d):
        X = pd.DataFrame(index=d.index)
        lb = np.log1p(d["BudgetUSD"])
        year = d["ReleaseYear"].fillna(d["ReleaseDate"].dt.year)
        X["log_budget"] = lb
        X["budget_missing"] = d["BudgetUSD"].isna().astype(int)
        X["budget_vs_year_median"] = lb - year.map(YEAR_BUDGET_MED).fillna(GLOBAL_BUDGET_MED)   # koreksi inflasi budget
        X["ReleaseYear"] = year
        dt = d["ReleaseDate"].dt
        X["month"], X["quarter"], X["dayofweek"], X["dayofyear"] = dt.month, dt.quarter, dt.dayofweek, dt.dayofyear
        X["is_summer"] = X["month"].isin([5, 6, 7]).astype(int)
        X["is_holiday_season"] = X["month"].isin([11, 12]).astype(int)
        X["is_friday_release"] = (X["dayofweek"] == 4).astype(int)
        gl = d["Genre"].apply(parse_genres)
        first = gl.str[0]
        X["n_genres"] = gl.str.len()
        X["primary_genre"] = pd.Categorical(first.where(first.isin(PRIMARY_CATS), "Other"), categories=PRIMARY_CATS)
        for g in TOP_GENRES:
            X["genre_" + san(g)] = gl.apply(lambda l, g=g: int(g in l))
        ctry = d["Country"].fillna("Unknown")
        X["country_grp"] = pd.Categorical(ctry.where(ctry.isin(COUNTRY_CATS), "Other"), categories=COUNTRY_CATS)
        X["is_usa"] = ctry.str.contains(r"\b(?:USA|US|United States)\b", case=False, regex=True).astype(int)
        X["director_freq"] = d["Director"].fillna("Unknown").map(DIRECTOR_FREQ).fillna(0)       # popularitas/pengalaman
        X["actor_freq"] = d["LeadActor"].fillna("Unknown").map(ACTOR_FREQ).fillna(0)
        return X

    X_base = build_base_features(df)
    keys = df[["Director", "LeadActor"]].fillna("Unknown")
    print(f"{X_base.shape[1]} fitur dasar + 2 target-encoding (director_te, actor_te dibuat per-target, out-of-fold)")
    print("Fitur:", list(X_base.columns))
    display(X_base.head())

    def target_encode(k_tr, y_tr, m=10, n_splits=5):
        """Smoothed target encoding. Train memakai out-of-fold agar tidak bocor; valid/test memakai encoder dari train."""
        gm = y_tr.mean()
        oof = pd.Series(gm, index=k_tr.index, dtype=float)
        for a, b in KFold(n_splits, shuffle=True, random_state=SEED).split(k_tr):
            g = y_tr.iloc[a].groupby(k_tr.iloc[a]).agg(["sum", "count"])
            enc = (g["sum"] + m * gm) / (g["count"] + m)
            oof.iloc[b] = k_tr.iloc[b].map(enc).fillna(gm).values
        g = y_tr.groupby(k_tr).agg(["sum", "count"])
        return oof, (g["sum"] + m * gm) / (g["count"] + m), gm


    # =====================================================================================
    # 7. TRAINING LIGHTGBM (1 model per target, early stopping pada validation set)
    # =====================================================================================
    section("7. TRAINING LIGHTGBM")
    def make_model(n_train):
        return lgb.LGBMRegressor(
            objective="regression", n_estimators=5000, learning_rate=0.03,
            num_leaves=15, max_depth=6, min_child_samples=max(10, int(0.01 * n_train)),
            subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            reg_alpha=0.5, reg_lambda=2.0, cat_smooth=20, min_data_per_group=20, cat_l2=10,
            random_state=SEED, n_jobs=-1, verbose=-1)

    models, curves, importances, test_raw, rows = {}, {}, {}, {}, []
    for tgt, kind in TARGETS.items():
        y_all = np.log1p(df[tgt]) if kind == "money" else df[tgt].astype(float)
        tr_i = idx_tr[y_all.loc[idx_tr].notna().values]
        va_i = idx_va[y_all.loc[idx_va].notna().values]
        if len(tr_i) < 50 or len(va_i) < 10:
            print(f"[SKIP] {tgt}: data berlabel terlalu sedikit"); continue
        ytr, yva = y_all.loc[tr_i], y_all.loc[va_i]
        Xtr, Xva, Xte = X_base.loc[tr_i].copy(), X_base.loc[va_i].copy(), X_base.loc[idx_te].copy()

        te_maps = {}
        for col, name in [("Director", "director_te"), ("LeadActor", "actor_te")]:
            oof, enc, gm = target_encode(keys.loc[tr_i, col], ytr)
            Xtr[name] = oof.values
            Xva[name] = keys.loc[va_i, col].map(enc).fillna(gm).values
            Xte[name] = keys.loc[idx_te, col].map(enc).fillna(gm).values
            te_maps[name] = (col, enc, gm)

        model = make_model(len(Xtr))
        model.fit(Xtr, ytr, eval_set=[(Xtr, ytr), (Xva, yva)], eval_names=["train", "valid"], eval_metric="rmse",
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        best = int(model.best_iteration_ or model.n_estimators_)

        for split, X_, i_ in [("train", Xtr, tr_i), ("valid", Xva, va_i)]:
            rows.append({"target": tgt, "split": split, "n": len(i_),
                         **metrics(df.loc[i_, tgt], to_orig(model.predict(X_), tgt, kind), kind)})
        models[tgt] = dict(model=model, features=list(Xtr.columns), te_maps=te_maps, kind=kind, best_iter=best)
        curves[tgt] = model.evals_result_
        importances[tgt] = pd.Series(model.booster_.feature_importance("gain"), index=Xtr.columns).sort_values(ascending=False)
        test_raw[tgt] = to_orig(model.predict(Xte), tgt, kind)
        print(f"[OK] {tgt:<22} train={len(tr_i):>5} valid={len(va_i):>5} | best_iter={best:>4} | "
              f"R2 valid={rows[-1]['R2']:.3f}")


    # =====================================================================================
    # 8. EVALUASI (train vs valid vs test) + cek overfitting
    # =====================================================================================
    section("8. EVALUASI MODEL")
    def apply_order(p):
        for big, small in ORDER_RULES:
            if big in p and small in p:
                p[big] = np.maximum(p[big], p[small])
        return p

    test_pred = apply_order(dict(test_raw))           # test = data yang tidak pernah disentuh saat training/tuning
    for tgt, arr in test_pred.items():
        m = df.loc[idx_te, tgt].notna().values
        rows.append({"target": tgt, "split": "test", "n": int(m.sum()),
                     **metrics(df.loc[idx_te, tgt].values[m], arr[m], TARGETS[tgt])})

    res = pd.DataFrame(rows)
    res["split"] = pd.Categorical(res["split"], ["train", "valid", "test"])
    res = res.sort_values(["target", "split"]).reset_index(drop=True)
    print("R2 & RMSE: skala log1p untuk target USD | MAE & MedAPE_%: skala asli (USD / poin rating)")
    display(res)

    gap = res.pivot(index="target", columns="split", values="R2")
    gap["gap_train-test"] = gap["train"] - gap["test"]
    gap["status"] = np.where(gap["gap_train-test"] > 0.30, "cek overfitting", "OK")
    print("Cek overfitting (selisih R2 train vs test):")
    display(gap)

    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    for ax in axes.ravel()[len(curves):]:
        ax.axis("off")
    for ax, (t, ev) in zip(axes.ravel(), curves.items()):
        ax.plot(ev["train"]["rmse"], label="train"); ax.plot(ev["valid"]["rmse"], label="valid")
        ax.axvline(models[t]["best_iter"], ls="--", c="gray", label="best iter")
        ax.set_title(f"Learning curve - {t}"); ax.set_xlabel("iterasi"); ax.set_ylabel("RMSE"); ax.legend()
    plt.tight_layout(); plt.show()


    # =====================================================================================
    # 9. FEATURE IMPORTANCE (gain)
    # =====================================================================================
    section("9. FEATURE IMPORTANCE")
    fig, axes = plt.subplots(2, 3, figsize=(20, 11))
    for ax in axes.ravel()[len(importances):]:
        ax.axis("off")
    for ax, (t, s) in zip(axes.ravel(), importances.items()):
        top = (s / s.sum() * 100).head(12)
        sns.barplot(x=top.values, y=top.index, ax=ax, color="#3b7dd8")
        ax.set_title(f"Top fitur - {t}"); ax.set_xlabel("% gain")
    plt.tight_layout(); plt.show()

    imp_df = pd.DataFrame({t: s / s.sum() * 100 for t, s in importances.items()}).fillna(0)
    imp_df["rata_rata"] = imp_df.mean(axis=1)
    imp_df = imp_df.sort_values("rata_rata", ascending=False)
    print("Importance (% gain) gabungan semua target - 15 teratas:")
    display(imp_df.head(15))


    # =====================================================================================
    # 10. OUTPUT PREDIKSI
    # =====================================================================================
    section("10. OUTPUT PREDIKSI")
    # 10a. Prediksi pada TEST SET (data yang tidak pernah dilihat model)
    info_cols = [c for c in ["MovieID", "Title", "Genre", "ReleaseYear", "BudgetUSD"] if c in df]
    test_out = df.loc[idx_te, info_cols].copy()
    for t, arr in test_pred.items():
        test_out["Actual_" + t] = df.loc[idx_te, t].values
        test_out["Pred_" + t] = np.round(arr, 2 if TARGETS[t] == "rating" else 0)
    print("Contoh prediksi vs aktual pada test set:")
    display(test_out.head(15))

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax in axes.ravel()[len(test_pred):]:
        ax.axis("off")
    for ax, (t, arr) in zip(axes.ravel(), test_pred.items()):
        a = df.loc[idx_te, t].values; m = ~np.isnan(a)
        if TARGETS[t] == "money":
            x, y = np.log10(a[m] + 1), np.log10(arr[m] + 1); ax.set_xlabel("Aktual log10(USD)"); ax.set_ylabel("Prediksi log10(USD)")
        else:
            x, y = a[m], arr[m]; ax.set_xlabel("Aktual"); ax.set_ylabel("Prediksi")
        ax.scatter(x, y, alpha=.4, s=14)
        lim = [min(x.min(), y.min()), max(x.max(), y.max())]; ax.plot(lim, lim, "r--", lw=1)
        ax.set_title(f"{t} (test)")
    plt.tight_layout(); plt.show()

    # 10b. Fungsi prediksi untuk FILM BARU
    def predict_movie(budget, genre, release_date, country, director, lead_actor):
        row = pd.DataFrame([{"BudgetUSD": float(budget), "Genre": genre, "ReleaseDate": pd.to_datetime(release_date, errors="coerce"),
                             "Country": country, "Director": director, "LeadActor": lead_actor}])
        row["ReleaseYear"] = row["ReleaseDate"].dt.year
        Xb = build_base_features(row)
        kk = row[["Director", "LeadActor"]].fillna("Unknown")
        out = {}
        for t, art in models.items():
            X = Xb.copy()
            for name, (col, enc, gm) in art["te_maps"].items():
                X[name] = kk[col].map(enc).fillna(gm).values
            out[t] = float(to_orig(art["model"].predict(X[art["features"]]), t, art["kind"])[0])
        out = apply_order(out)
        return {t: (round(float(v), 2) if TARGETS[t] == "rating" else round(float(v))) for t, v in out.items()}

    mode_of = lambda c: df[c].mode().iloc[0] if df[c].notna().any() else "Unknown"
    demo = dict(genre=mode_of("Genre"), country=mode_of("Country"), director=mode_of("Director"),
                lead_actor=mode_of("LeadActor"), release_date=f"{int(df['ReleaseYear'].max())}-07-04")
    base_budget = float(df["BudgetUSD"].median())
    print("Contoh film baru ->", {**demo, "budget": round(base_budget)})
    print("Simulasi dampak budget terhadap prediksi (fitur lain tetap):")
    display(pd.DataFrame([{"BudgetUSD": round(base_budget * k), **predict_movie(budget=base_budget * k, **demo)}
                          for k in (0.5, 1, 2, 4)]))

    # 10c. Simpan hasil
    test_out.to_csv(f"{OUT_DIR}/test_predictions.csv", index=False)
    res.to_csv(f"{OUT_DIR}/model_metrics.csv", index=False)
    imp_df.to_csv(f"{OUT_DIR}/feature_importance.csv")
    joblib.dump({"models": models,
                 "meta": dict(TOP_GENRES=TOP_GENRES, PRIMARY_CATS=PRIMARY_CATS, COUNTRY_CATS=COUNTRY_CATS,
                              YEAR_BUDGET_MED=YEAR_BUDGET_MED, GLOBAL_BUDGET_MED=GLOBAL_BUDGET_MED,
                              DIRECTOR_FREQ=DIRECTOR_FREQ, ACTOR_FREQ=ACTOR_FREQ, ORDER_RULES=ORDER_RULES, BOUNDS=BOUNDS)},
                f"{OUT_DIR}/movie_lgbm_models.joblib")
    print(f"\nTersimpan di {OUT_DIR}: test_predictions.csv, model_metrics.csv, feature_importance.csv, movie_lgbm_models.joblib")
    print("SELESAI.")